# 🔍 Rumor Detection — Evaluation Notebook
### Bidirectional LSTM with Graph-Based Propagation Features

This notebook walks through the complete pipeline:
1. Generate / load data
2. Build and train the dual-branch BiLSTM model
3. Full classification evaluation (precision / recall / F1 / confusion matrix)
4. **Early-detection curve** — F1 vs. time since source post (the headline result)
5. Attention weight visualization
6. Ablation study: text-only vs. propagation-only vs. full model


## 1. Environment Setup

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import tensorflow as tf
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, ConfusionMatrixDisplay
)

# Make project root importable (works whether notebook is inside the repo root
# or inside a 'notebooks/' subfolder)
repo_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"TensorFlow  : {tf.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"Repo root   : {repo_root}")


## 2. Configuration

In [ ]:
import config

print("Label mode  :", config.LABEL_MODE)
print("Num classes :", config.NUM_CLASSES)
print("Max seq len :", config.MAX_SEQ_LEN)
print("Max prop len:", config.MAX_PROP_LEN)
print("Prop features:", config.NUM_PROP_FEATURES)
print("Early detection checkpoints (min):", config.EARLY_DETECTION_DEADLINES_MIN)


## 3. Data: Generate Synthetic Demo Data

If you have a real PHEME / Twitter15-16 dataset ready, skip this cell and
point `DATASET_PATH` at your converted JSON file instead.


In [ ]:
from data.generate_synthetic_data import generate_dataset

DATASET_PATH = os.path.join(config.PROCESSED_DATA_DIR, "synthetic_dataset.json")

if not os.path.exists(DATASET_PATH):
    print("Generating synthetic dataset...")
    generate_dataset(n_stories=1200, out_path=DATASET_PATH)
else:
    print(f"Dataset already exists at {DATASET_PATH}")


### 3.1 Inspect label distribution

In [ ]:
from src.data_loader import load_stories
from collections import Counter

stories = load_stories(DATASET_PATH)
print(f"Total stories: {len(stories)}")

label_counts = Counter(s["label"] for s in stories)
for lbl, cnt in sorted(label_counts.items()):
    print(f"  {lbl:<12}: {cnt:>4}  ({cnt/len(stories)*100:.1f}%)")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
labels, counts = zip(*sorted(label_counts.items()))
bars = ax.bar(labels, counts, color=["#4c72b0","#55a868","#c44e52","#8172b2"])
ax.bar_label(bars, padding=3)
ax.set_title("Label Distribution")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_DIR, "label_distribution.png"), dpi=150)
plt.show()


### 3.2 Propagation stats by label

In [ ]:
import pandas as pd

rows = []
for s in stories:
    evts = s["events"]
    rows.append({
        "label": s["label"],
        "n_events": len(evts),
        "max_depth": max((e["depth"] for e in evts), default=0),
        "max_delay_min": max((e["time_delay_min"] for e in evts), default=0),
        "pct_retweet": sum(1 for e in evts if e["type"]=="retweet") / max(len(evts),1),
        "avg_followers": np.mean([e["followers_count"] for e in evts]) if evts else 0,
    })

df = pd.DataFrame(rows)
print(df.groupby("label")[["n_events","max_depth","max_delay_min","avg_followers"]].mean().round(2))


## 4. Preprocessing

In [ ]:
from src.preprocessing import fit_tokenizer, save_tokenizer, build_tensors, train_val_test_split

train_stories, val_stories, test_stories = train_val_test_split(stories)
print(f"Train: {len(train_stories)}  Val: {len(val_stories)}  Test: {len(test_stories)}")

tokenizer = fit_tokenizer(train_stories)
save_tokenizer(tokenizer)
vocab_size = min(config.MAX_VOCAB_SIZE, len(tokenizer.word_index) + 1)
print(f"Vocabulary size: {vocab_size}")

X_text_train, X_prop_train, y_train, _ = build_tensors(train_stories, tokenizer)
X_text_val,   X_prop_val,   y_val,   _ = build_tensors(val_stories,   tokenizer)
X_text_test,  X_prop_test,  y_test,  _ = build_tensors(test_stories,  tokenizer)

print(f"X_text_train shape : {X_text_train.shape}")
print(f"X_prop_train shape : {X_prop_train.shape}")
print(f"y_train shape      : {y_train.shape}")


## 5. Build Model

In [ ]:
from src.model import build_model, AttentionPool

model = build_model(vocab_size=vocab_size)
model.summary()


## 6. Training

In [ ]:
np.random.seed(config.RANDOM_SEED)
tf.random.set_seed(config.RANDOM_SEED)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(config.MODEL_DIR, "best_model.keras"),
        monitor="val_loss", save_best_only=True, verbose=0
    ),
]

history = model.fit(
    [X_text_train, X_prop_train], y_train,
    validation_data=([X_text_val, X_prop_val], y_val),
    epochs=config.EPOCHS,
    batch_size=config.BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)


### 6.1 Training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history["loss"],     label="Train loss")
ax1.plot(history.history["val_loss"], label="Val loss")
ax1.set_title("Loss")
ax1.set_xlabel("Epoch")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(history.history["accuracy"],     label="Train acc")
ax2.plot(history.history["val_accuracy"], label="Val acc")
ax2.set_title("Accuracy")
ax2.set_xlabel("Epoch")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_DIR, "training_curves.png"), dpi=150)
plt.show()


## 7. Full Evaluation

In [ ]:
y_probs = model.predict([X_text_test, X_prop_test], verbose=0)
y_pred  = np.argmax(y_probs, axis=1)

label_map = config.LABEL_MAP_4CLASS if config.LABEL_MODE == "fourclass" else config.LABEL_MAP_BINARY
label_names = [k for k, v in sorted(label_map.items(), key=lambda x: x[1])]

print("=" * 60)
print("CLASSIFICATION REPORT (Full propagation)")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=label_names, digits=4))

macro_f1 = f1_score(y_test, y_pred, average="macro")
print(f"Macro-F1 : {macro_f1:.4f}")


### 7.1 Confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — Full propagation")
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_DIR, "confusion_matrix.png"), dpi=150)
plt.show()


## 8. Early Detection Curve

This is the headline result of the project. We truncate the propagation
sequence at each deadline and re-run inference to simulate what the model
would predict if it had to decide at that point in time.


In [ ]:
from src.preprocessing import build_tensors

deadlines = config.EARLY_DETECTION_DEADLINES_MIN
macro_f1s, accs, avg_events = [], [], []

print(f"{'Deadline':>10} | {'Avg events':>10} | {'Macro-F1':>9} | {'Accuracy':>9}")
print("-" * 50)

for dl in deadlines:
    Xt, Xp, yt, ne = build_tensors(test_stories, tokenizer, deadline_min=dl)
    probs = model.predict([Xt, Xp], verbose=0)
    yp    = np.argmax(probs, axis=1)
    mf1   = f1_score(yt, yp, average="macro")
    acc   = (yp == yt).mean()
    macro_f1s.append(mf1)
    accs.append(acc)
    avg_events.append(ne.mean())
    print(f"{dl:>9}m | {ne.mean():>10.1f} | {mf1:>9.4f} | {acc:>9.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(deadlines, macro_f1s, marker="o", linewidth=2, label="Macro-F1")
ax.plot(deadlines, accs,      marker="s", linewidth=2, label="Accuracy")
ax.set_xscale("log")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"{int(x)}m" if x < 60 else f"{int(x//60)}h{int(x%60):02d}m" if x % 60 else f"{int(x//60)}h"
))
ax.set_xlabel("Detection deadline (log scale)")
ax.set_ylabel("Score")
ax.set_title("Early Rumor Detection Performance vs. Time", fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_DIR, "early_detection_curve.png"), dpi=150)
plt.show()
print("Saved: outputs/early_detection_curve.png")


## 9. Attention Weight Visualization

In [ ]:
# Build an auxiliary model that also outputs attention weights
attn_model = tf.keras.Model(
    inputs=model.input,
    outputs=[
        model.output,
        model.get_layer("text_attention").output[1],   # (batch, seq_len)
        model.get_layer("prop_attention").output[1],   # (batch, prop_len)
    ],
    name="AttentionModel"
)

# Pick one example per class and visualize
n_viz = 4
fig, axes = plt.subplots(n_viz, 2, figsize=(14, 3 * n_viz))

for row, class_id in enumerate(range(n_viz)):
    idxs = np.where(y_test == class_id)[0]
    if len(idxs) == 0:
        continue
    i = idxs[0]

    probs_i, text_attn, prop_attn = attn_model.predict(
        [X_text_test[i:i+1], X_prop_test[i:i+1]], verbose=0
    )
    pred_label = label_names[np.argmax(probs_i[0])]
    true_label = label_names[y_test[i]]

    # Text attention
    idx2word = {v: k for k, v in tokenizer.word_index.items()}
    tokens = [idx2word.get(t, "<PAD>") for t in X_text_test[i] if t != 0]
    attn_vals = text_attn[0, :len(tokens)]

    ax = axes[row, 0]
    ax.barh(range(len(tokens)), attn_vals[::-1], color="#4c72b0")
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens[::-1], fontsize=8)
    ax.set_title(f"Text attention | true={true_label} pred={pred_label}", fontsize=9)

    # Propagation attention
    n_prop = (X_prop_test[i].sum(axis=1) != 0).sum()
    p_attn = prop_attn[0, :n_prop]
    ax2 = axes[row, 1]
    ax2.plot(range(n_prop), p_attn, marker="o", markersize=4, color="#c44e52")
    ax2.set_xlabel("Propagation event index (time ordered)")
    ax2.set_ylabel("Attention weight")
    ax2.set_title(f"Propagation attention | true={true_label} pred={pred_label}", fontsize=9)
    ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_DIR, "attention_visualization.png"), dpi=150)
plt.show()


## 10. Ablation Study

Compare the full dual-branch model against:
- **Text-only**: propagation input is zeroed out (branch contributes nothing)
- **Propagation-only**: text input is zeroed out
- **Full model**: both branches active (default)


In [ ]:
def eval_variant(name, X_text, X_prop, y_true):
    probs = model.predict([X_text, X_prop], verbose=0)
    y_pred = np.argmax(probs, axis=1)
    f1 = f1_score(y_true, y_pred, average="macro")
    acc = (y_pred == y_true).mean()
    print(f"  {name:<30} Macro-F1: {f1:.4f}  Acc: {acc:.4f}")
    return f1, acc

print("Ablation Study — Test Set")
print("-" * 55)

# Full model
f1_full, acc_full = eval_variant("Full model (text + propagation)",
    X_text_test, X_prop_test, y_test)

# Text-only: zero out propagation input
X_prop_zeros = np.zeros_like(X_prop_test)
f1_text, acc_text = eval_variant("Text-only (prop zeroed out)",
    X_text_test, X_prop_zeros, y_test)

# Prop-only: zero out text input
X_text_zeros = np.zeros_like(X_text_test)
f1_prop, acc_prop = eval_variant("Prop-only (text zeroed out)",
    X_text_zeros, X_prop_test, y_test)


In [ ]:
variants = ["Text-only", "Prop-only", "Full model"]
f1_vals  = [f1_text, f1_prop, f1_full]
colors   = ["#4c72b0", "#c44e52", "#55a868"]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(variants, f1_vals, color=colors, width=0.5)
ax.bar_label(bars, fmt="%.4f", padding=3)
ax.set_ylim(0, max(f1_vals) * 1.2)
ax.set_ylabel("Macro-F1")
ax.set_title("Ablation Study — Macro-F1 Comparison")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_DIR, "ablation_study.png"), dpi=150)
plt.show()


## 11. Save Final Model

In [ ]:
final_path = os.path.join(config.MODEL_DIR, "final_model.keras")
model.save(final_path)
print(f"Model saved: {final_path}")

# Save test stories for reproducible re-evaluation
test_out = os.path.join(config.PROCESSED_DATA_DIR, "test_stories.json")
with open(test_out, "w") as f:
    json.dump(test_stories, f)
print(f"Test split saved: {test_out}")


## 12. Summary

In [ ]:
print("=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"  Full model — Macro-F1 : {f1_full:.4f}")
print(f"  Full model — Accuracy : {acc_full:.4f}")
print()
print("  Early detection (Macro-F1):")
for dl, mf1 in zip(deadlines, macro_f1s):
    label = f"{dl}m" if dl < 60 else f"{dl//60}h"
    print(f"    ≤ {label:<5} : {mf1:.4f}")
print()
print("  Ablation:")
print(f"    Text-only  : {f1_text:.4f}")
print(f"    Prop-only  : {f1_prop:.4f}")
print(f"    Full model : {f1_full:.4f}")
print()
print("  Output files saved to:", config.OUTPUT_DIR)


---
*Rumor Detection on Twitter — BiLSTM with Graph-Based Propagation Features*
*Sri Krishna College of Technology | B.E. CSE (AI & ML)*
